<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=312711644" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [1]:
import json
import os
from collections import Counter, deque

is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

# ================================
# LOAD DATA
# ================================
if is_rerun:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
else:
    path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

with open(path) as f:
    data = json.load(f)

# ================================
# BASIC OPS
# ================================
def copy_grid(g): return [row[:] for row in g]

def rotate90(g): return list(map(list, zip(*g[::-1])))

def most_common_color(grid):
    flat = [c for row in grid for c in row]
    return Counter(flat).most_common(1)[0][0]

# ================================
# OBJECT DETECTION (BFS)
# ================================
def find_objects(grid):
    h, w = len(grid), len(grid[0])
    visited = [[False]*w for _ in range(h)]
    objects = []

    for i in range(h):
        for j in range(w):
            if grid[i][j] != 0 and not visited[i][j]:
                q = deque([(i, j)])
                visited[i][j] = True
                cells = []

                while q:
                    x, y = q.popleft()
                    cells.append((x, y))

                    for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nx, ny = x+dx, y+dy
                        if 0 <= nx < h and 0 <= ny < w:
                            if grid[nx][ny] != 0 and not visited[nx][ny]:
                                visited[nx][ny] = True
                                q.append((nx, ny))

                objects.append(cells)

    return objects

# ================================
# BOUNDING BOX
# ================================
def extract_object(grid, cells):
    xs = [x for x, y in cells]
    ys = [y for x, y in cells]

    minx, maxx = min(xs), max(xs)
    miny, maxy = min(ys), max(ys)

    obj = []
    for i in range(minx, maxx+1):
        row = []
        for j in range(miny, maxy+1):
            if (i, j) in cells:
                row.append(grid[i][j])
            else:
                row.append(0)
        obj.append(row)

    return obj

# ================================
# PLACE OBJECT
# ================================
def place_object(grid, obj, top, left):
    h, w = len(grid), len(grid[0])
    oh, ow = len(obj), len(obj[0])

    new = [row[:] for row in grid]

    for i in range(oh):
        for j in range(ow):
            if obj[i][j] != 0:
                x, y = top+i, left+j
                if 0 <= x < h and 0 <= y < w:
                    new[x][y] = obj[i][j]

    return new

# ================================
# FIND OBJECT SHIFT RULE
# ================================
def find_shift(train_pairs):
    shifts = []

    for pair in train_pairs:
        inp = pair["input"]
        out = pair["output"]

        objs_in = find_objects(inp)
        objs_out = find_objects(out)

        if len(objs_in) == 1 and len(objs_out) == 1:
            obj_in = objs_in[0]
            obj_out = objs_out[0]

            dx = obj_out[0][0] - obj_in[0][0]
            dy = obj_out[0][1] - obj_in[0][1]

            shifts.append((dx, dy))

    if shifts and all(s == shifts[0] for s in shifts):
        return shifts[0]

    return None

# ================================
# SOLVER
# ================================
def solve_task(task):
    train = task["train"]
    test_input = task["test"][0]["input"]

    predictions = []

    # Object movement
    shift = find_shift(train)

    if shift:
        objs = find_objects(test_input)
        if len(objs) == 1:
            obj = extract_object(test_input, objs[0])

            h, w = len(test_input), len(test_input[0])
            base = [[0]*w for _ in range(h)]

            dx, dy = shift
            x0, y0 = objs[0][0]

            new_grid = place_object(base, obj, x0+dx, y0+dy)
            predictions.append(new_grid)

    #  Rotation fallback
    predictions.append(rotate90(test_input))

    #  Copy fallback
    predictions.append(copy_grid(test_input))

    # ensure 2
    return predictions[:2]

# ================================
# BUILD SUBMISSION
# ================================
submission = []

for task_id, task in data.items():
    preds = solve_task(task)

    submission.append({
        "task_id": task_id,
        "attempts": preds
    })

# ================================
# SAVE
# ================================
with open("submission.json", "w") as f:
    json.dump(submission, f)

print("version 22 (OBJECT AI) READY!")

version 22 (OBJECT AI) READY!
